<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

## 02-2 · Part 4: Level Sets and Feasibility

**Level sets describe equal output, while the feasible set contains decisions satisfying every requirement.**

Part 3 compared geometric step budgets. This unit returns to the classroom simulation from Part 1 and uses the same decision \\(u=(u_{\mathrm{early}},u_{\mathrm{late}})\\). Energy and score contours are compared with the physical requirements, with \\(\lambda_E=1\\).

### 1 · Classroom formulation

The real decision is how much cooling to use early and late. More cooling can reduce heat discomfort, but it uses energy and may make the room too cold.

Let \\(u=(u_{\mathrm{early}},u_{\mathrm{late}})\\). The horizon has \\(n=12\\) decision steps, with actions \\(u_0,\ldots,u_{11}\\) and states \\(T_0,\ldots,T_{12}\\).

> $\displaystyle u_t=\begin{cases}u_{\mathrm{early}},&t=0,\ldots,5,\\u_{\mathrm{late}},&t=6,\ldots,11.\end{cases}$

Indoor temperature \\(T_t\\) is produced by the system; it is not chosen directly. With initial temperature \\(T_0=27\,^{\circ}\mathrm C\\), the physical transition is

> $\displaystyle T_{t+1}=F(T_t,T_t^{\mathrm{out}},N_t,u_t;a,b,c)$
>
> $\displaystyle \phantom{T_{t+1}}=T_t+a(T_t^{\mathrm{out}}-T_t)+bN_t-cu_t,\quad t=0,\ldots,11.$

| Role | Values held fixed in the demonstrations |
|:---|:---|
| Fixed parameters | $(a,b,c)=(0.12,0.012,0.45)$ |
| External inputs | $T_t^{\mathrm{out}}=31\,^{\circ}\mathrm C$ and $N_t=20$ people at every step |
| Cooling limits | $u_{\min}=0$, $u_{\max}=5$ cooling units |
| State limits | $T_{\min}=20\,^{\circ}\mathrm C$, $T_{\max}=30\,^{\circ}\mathrm C$ |
| Energy limit | $E_{\max}=60$ model energy units |
| Energy-weight hyperparameter | $\lambda_E=1$ unless stated otherwise |

The performance mapping \\(G\\) measures discomfort outside the 22–24 °C comfort range and energy use:

> $\displaystyle D(u)=\sum_{t=1}^{12}\left[\max(T_t-24,0)^2+\max(22-T_t,0)^2\right].$
>
> $\displaystyle E(u)=\frac12\sum_{t=0}^{11}u_t^2=3\left(u_{\mathrm{early}}^2+u_{\mathrm{late}}^2\right).$

\\(D\\) uses squared-temperature step units. \\(E\\) uses model energy units, not calibrated kWh. The weight converts energy into the chosen score scale:

> $\displaystyle J(u;\lambda_E)=H(D(u),E(u);\lambda_E)=D(u)+\lambda_EE(u).$

Feasibility requires cooling bounds, all state limits for \\(t=1,\ldots,12\\), and \\(E(u)\le E_{\max}\\). The comfort range is a performance target; the wider 20–30 °C range is a hard requirement.

In standard notation, \\(x=[u_{\mathrm{early}},u_{\mathrm{late}}]^{\mathsf T}\\) and \\(y=\operatorname{Sim}(x)\\). Here \\(\operatorname{Sim}\\) composes repeated \\(F\\) transitions with \\(G\\). The objective \\(f(y;\lambda_E)\\) is the standard-form name for the score supplied by \\(H\\). The score depends on the cooling decision through both the simulated states and energy use.

In [ ]:
import sys
import warnings

import matplotlib
import numpy as np
from matplotlib.patches import Patch, Rectangle
from matplotlib.lines import Line2D


def _pyplot(*, interactive=False):
    """Use ipympl outside the Playground, with a static fallback."""
    if interactive and sys.platform != "emscripten":
        try:
            matplotlib.use("widget", force=True)
        except (ImportError, RuntimeError, ValueError):
            try:
                matplotlib.use("module://ipympl.backend_nbagg", force=True)
            except (ImportError, RuntimeError, ValueError):
                warnings.warn("Interactive backend unavailable; showing a static preview.")
    import matplotlib.pyplot as plt
    return plt


# Horizon and initial state
TIME_STEPS = 12
INITIAL_TEMPERATURE = 27.0

# Fixed parameters
WEATHER_EXCHANGE = 0.12
OCCUPANT_HEAT = 0.012
COOLING_EFFECT = 0.45

# External inputs
OUTSIDE_TEMPERATURE = np.full(TIME_STEPS, 31.0)
OCCUPANTS = np.full(TIME_STEPS, 20.0)

# Requirement limits
MIN_COOLING, MAX_COOLING = 0.0, 5.0
MIN_TEMPERATURE, MAX_TEMPERATURE = 20.0, 30.0
MAX_ENERGY = 60.0

# Evaluation hyperparameter
ENERGY_WEIGHT = 1.0

# Stable visual roles
BLUE, TEAL, ORANGE = "#2563EB", "#0F8B7C", "#E88726"
PURPLE, GRAY = "#7C3AED", "#9CA3AF"


def style_axis(axis):
    axis.grid(alpha=0.25)
    axis.spines[["top", "right"]].set_visible(False)


def decision_axes(axis):
    axis.set(xlabel="Early cooling (cooling units)",
             ylabel="Late cooling (cooling units)", xlim=(0, 5), ylim=(0, 5))
    axis.set_aspect("equal")
    style_axis(axis)

Simulation produces the state path. The performance mapping calculates discomfort and energy; the requirement checks determine feasibility. Evaluation collects these quantities with the score in one result.


In [ ]:
def expand_decision(decision):
    """Expand the chosen levels into u_0, ..., u_11."""
    decision = np.asarray(decision, dtype=float)
    if decision.shape != (2,) or not np.isfinite(decision).all():
        raise ValueError("A decision must contain two finite cooling levels.")
    return np.repeat(decision, TIME_STEPS // 2)


def simulate_classroom(decision):
    cooling_schedule = expand_decision(decision)
    temperatures = np.empty(TIME_STEPS + 1)
    temperatures[0] = INITIAL_TEMPERATURE
    for t in range(TIME_STEPS):
        temperatures[t + 1] = (
            temperatures[t]
            + WEATHER_EXCHANGE * (OUTSIDE_TEMPERATURE[t] - temperatures[t])
            + OCCUPANT_HEAT * OCCUPANTS[t]
            - COOLING_EFFECT * cooling_schedule[t]
        )
    return cooling_schedule, temperatures


def performance_outputs(cooling_schedule, temperatures):
    discomfort = np.sum(
        np.maximum(temperatures[1:] - 24.0, 0.0) ** 2
        + np.maximum(22.0 - temperatures[1:], 0.0) ** 2
    )
    energy = 0.5 * np.sum(cooling_schedule ** 2)
    return float(discomfort), float(energy)


def check_feasibility(decision, temperatures, energy):
    violations = []
    if not np.all((MIN_COOLING <= decision) & (decision <= MAX_COOLING)):
        violations.append("cooling bound")
    if np.min(temperatures[1:]) < MIN_TEMPERATURE:
        violations.append("minimum temperature")
    if np.max(temperatures[1:]) > MAX_TEMPERATURE:
        violations.append("maximum temperature")
    if energy > MAX_ENERGY:
        violations.append("energy limit")
    return tuple(violations)


def evaluate_candidate(decision, energy_weight=ENERGY_WEIGHT):
    decision = np.asarray(decision, dtype=float)
    cooling_schedule, temperatures = simulate_classroom(decision)
    discomfort, energy = performance_outputs(cooling_schedule, temperatures)
    violations = check_feasibility(decision, temperatures, energy)
    return {
        "decision": decision.copy(), "cooling_schedule": cooling_schedule,
        "temperatures": temperatures, "discomfort": discomfort, "energy": energy,
        "feasible": not violations, "violations": violations,
        "energy_weight": float(energy_weight),
        "objective": discomfort + energy_weight * energy,
    }


def score(decision, energy_weight=ENERGY_WEIGHT):
    """Also defined outside the feasible set for geometry and derivatives."""
    return evaluate_candidate(decision, energy_weight)["objective"]

### 2 · Level sets and feasibility

A level set collects inputs with the same scalar output. In the cooling domain \\(\mathcal X=[0,5]^2\\), a score level set and a sublevel set are

> $\displaystyle \mathcal L_\tau=\{u\in\mathcal X:J(u;\lambda_E)=\tau\}.$
>
> $\displaystyle \mathcal S_\tau=\{u\in\mathcal X:J(u;\lambda_E)\le\tau\}.$

The scalar \\(\tau\\) is a score threshold. A labeled contour represents a level set. A sublevel set includes every point with score at most the threshold. A level set can also be a single point or be empty; it need not be a smooth curve.

The quadratic energy output has circular level sets:

> $\displaystyle E(u)=3\lVert u\rVert_2^2=\eta\quad\Longrightarrow\quad u_{\mathrm{early}}^2+u_{\mathrm{late}}^2=\eta/3.$

For a positive energy level \\(\eta\\), the full-plane contour is a circle of radius \\(\sqrt{\eta/3}\\). Only the portion inside the cooling domain appears below. The energy constraint \\(E\le60\\) gives a quarter disk inside the cooling bounds, but temperature requirements can reject further points.

The feasible set \\(\mathcal F\\) is the intersection of all requirements. **A level set describes equal output; feasibility describes requirements.** The set \\(\mathcal F\cap\mathcal S_\tau\\) contains feasible decisions meeting the score threshold.

Decisions \\((3,2)\\) and \\((2,3)\\) both have energy \\(E=39\\). Their different cooling schedules produce different temperature paths, discomfort values, and scores.

Energy and score contours are computed from a finite grid. The contours and region boundaries approximate the continuous geometry.

In [ ]:
def evaluate_grid(points_per_axis=81, energy_weight=ENERGY_WEIGHT):
    levels = np.linspace(MIN_COOLING, MAX_COOLING, points_per_axis)
    early, late = np.meshgrid(levels, levels)
    records = [evaluate_candidate((e, l), energy_weight)
               for e, l in zip(early.ravel(), late.ravel())]
    return {
        "early": early, "late": late, "records": records,
        "scores": np.array([r["objective"] for r in records]).reshape(early.shape),
        "energy": np.array([r["energy"] for r in records]).reshape(early.shape),
        "feasible": np.array([r["feasible"] for r in records]).reshape(early.shape),
        "energy_weight": energy_weight,
    }


def draw_score_map(axis, grid):
    axis.contourf(grid["early"], grid["late"], grid["feasible"].astype(int),
                  levels=[-0.5, 0.5, 1.5], colors=[GRAY, TEAL], alpha=0.16)
    contours = axis.contour(grid["early"], grid["late"], grid["scores"],
                            levels=[53, 55, 60, 75, 100, 150, 250, 400],
                            colors=BLUE, linewidths=1.0, alpha=0.85)
    # Place labels away from plot boundaries and the upper-right legend.
    label_positions = [(55, (3.25, 1.25)), (60, (3.9, 1.4)),
                       (75, (4.5, 2.2)), (100, (2.0, 1.1)),
                       (150, (1.7, 0.65)), (250, (1.0, 0.5)),
                       (400, (0.2, 0.3))]
    for level, position in label_positions:
        axis.clabel(contours, levels=[level], manual=[position], fontsize=8, fmt="%g")
    decision_axes(axis)


def show_level_sets(grid):
    plt = _pyplot()
    figure, axes = plt.subplots(1, 2, figsize=(10.8, 4.9))
    contours = axes[0].contour(grid["early"], grid["late"], grid["energy"],
                               levels=[12, 27, 39, 60, 90, 120], colors=BLUE)
    axes[0].clabel(contours, fmt=lambda v: f"E={v:g}", fontsize=8)
    axes[0].contourf(grid["early"], grid["late"], grid["energy"],
                     levels=[0, MAX_ENERGY], colors=[TEAL], alpha=0.13)
    axes[0].scatter([3, 2], [2, 3], color=ORANGE, s=60, zorder=5)
    axes[0].set_title("Equal energy follows circular contours")
    axes[0].legend(handles=[Patch(color=TEAL, alpha=0.2, label="Energy limit only")],
                    loc="upper left", fontsize=8)
    decision_axes(axes[0])
    draw_score_map(axes[1], grid)
    axes[1].scatter([3, 2], [2, 3], color=ORANGE, s=60, zorder=5)
    axes[1].set_title("Equal energy need not mean equal score")
    axes[1].legend(handles=[Patch(color=TEAL, alpha=0.2, label="All requirements met"),
                            Patch(color=GRAY, alpha=0.2, label="Infeasible")],
                    loc="upper right", fontsize=8)
    figure.suptitle(r"Score contours: $J=D+\lambda_E E$, fixed $\lambda_E=1$", color=PURPLE)
    figure.tight_layout(rect=(0, 0, 1, 0.95))
    return figure


def show_level_explorer(grid):
    plt = _pyplot(interactive=True)
    from matplotlib.widgets import Slider
    figure, axis = plt.subplots(figsize=(7.5, 6.4))
    figure.subplots_adjust(left=0.14, right=0.96, top=0.91, bottom=0.28)
    slider = Slider(figure.add_axes([0.26, 0.13, 0.58, 0.035]),
                    "Score threshold", 45, 140, valinit=60, valstep=1, color=PURPLE)
    note = figure.text(0.14, 0.035, "", fontsize=10)
    def update(_):
        axis.clear()
        threshold = slider.val
        axis.contourf(grid["early"], grid["late"], grid["feasible"].astype(int),
                      levels=[-0.5, 0.5, 1.5], colors=[GRAY, TEAL], alpha=0.12)
        accepted = grid["feasible"] & (grid["scores"] <= threshold)
        axis.scatter(grid["early"][accepted], grid["late"][accepted],
                      color=TEAL, s=5, alpha=0.7)
        if grid["scores"].min() < threshold < grid["scores"].max():
            axis.contour(grid["early"], grid["late"], grid["scores"],
                          levels=[threshold], colors=[PURPLE], linewidths=2)
        decision_axes(axis)
        axis.set_title("A score threshold filters the feasible set")
        axis.legend(handles=[Line2D([], [], color=PURPLE, lw=2, label="Equal-score contour"),
                             Patch(color=TEAL, alpha=0.65, label="Feasible and below threshold")],
                     loc="upper right", fontsize=8)
        note.set_text(f"{accepted.sum()} sampled decisions pass both tests; fixed energy weight = 1.\n"
                      "No passing samples does not prove the continuous set is empty.")
        figure.canvas.draw_idle()
    slider.on_changed(update)
    figure._sliders, figure._update = [slider], update
    update(None)
    return figure

The two orange decisions share an energy contour but lie on different score contours. Teal shading denotes the energy requirement on the left and the intersection of all requirements on the right.


<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/02-2_mathematics_for_optimization/assets/05_classroom_level_sets.svg" alt="Energy and score contours with two equal-energy decisions and the requirement regions" width="900" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

In [ ]:
classroom_grid = evaluate_grid()
for decision in [(3.0, 2.0), (2.0, 3.0)]:
    result = evaluate_candidate(decision)
    print(f"u={decision}, feasible={result['feasible']}, "
          f"E={result['energy']:.2f}, D={result['discomfort']:.2f}, "
          f"J={result['objective']:.2f}, T_12={result['temperatures'][-1]:.2f} °C")
level_figure = show_level_sets(classroom_grid)
_pyplot().show()
_pyplot().close(level_figure)

### 3 · Score thresholds

Increasing the score threshold \\(\tau\\) enlarges the sublevel set and its intersection with the feasible set. The purple contour can include infeasible points; darker teal samples satisfy both feasibility and the score threshold. The threshold changes set membership without changing the physical outcome of a fixed decision.

In [ ]:
level_explorer = show_level_explorer(classroom_grid)
_pyplot().show()